# MarkItUp — Sample Notebook

This notebook walks through everything MarkItUp can do: converting markdown to DOCX/PDF/HTML, working with themes, adding headers/footers/watermarks, and batch-converting the sample documents in this folder.

**Requirements:** `pip install "markitup-py[all]"`

In [ ]:
!pip install "markitup-py[all]"

In [ ]:
from markitup import MarkItUp, Theme
from markitup.theme import Running, Watermark, Banner
from markitup.fonts import SAFE_FONTS, list_fonts
import os

# Show what we're working with
print("Themes:", [t.replace('.yaml','') for t in os.listdir('../markitup/themes') if t.endswith('.yaml')])
print("Samples:", [f for f in os.listdir('.') if f.endswith('.md')])
print("Font categories:", list(SAFE_FONTS.keys()))

## 1. Quick Start — Convert a single file

Convert the academic paper sample to all three formats. The `report` theme is the default.

In [ ]:
m = MarkItUp(theme="report")
out_dir = "output"
os.makedirs(out_dir, exist_ok=True)

for ext in ("docx", "pdf", "html"):
    out = m.convert("academic-paper.md", f"{out_dir}/quickstart.{ext}")
    size_kb = os.path.getsize(out) / 1024
    print(f"✅ {ext:5s}  {size_kb:6.0f} KiB  → {out}")

## 2. Themes — Report vs Academic

Compare the two bundled themes side by side.

In [ ]:
for theme_name in ("report", "academic"):
    th = Theme.load(theme_name)
    print(f"📄 {theme_name}:")
    print(f"   body={th.fonts.body}, heading={th.fonts.heading}, mono={th.fonts.mono}")
    print(f"   base_size={th.type.base_size}pt, line_height={th.type.line_height}, ratio={th.type.ratio}")
    print(f"   text=#{th.colors.text}, heading=#{th.colors.heading}, accent=#{th.colors.accent}")
    print()

# Convert the same file with both themes
for theme_name in ("report", "academic"):
    m = MarkItUp(theme=theme_name)
    out = m.convert("academic-paper.md", f"{out_dir}/theme-{theme_name}.docx")
    print(f"✅ {theme_name} → {out}")

## 3. Headers, Footers & Watermarks

Configure running headers/footers on every page, plus a watermark and banner.

In [ ]:
th = Theme.load("report")

# Running header — left/center/right, with a divider line
th.header = Running(
    left="Upstride Labs Limited",
    center="CONFIDENTIAL — INTERNAL USE ONLY",
    right="Page {page} of {pages}",
    color="888888", size_pt=8.5, rule=True,
)

# Running footer
th.footer = Running(
    left="Confidential",
    center="Upstride Labs  |  Lagos  |  upstridelabs.com",
    right="{page} / {pages}",
    color="888888", size_pt=8.5, rule=True,
)

# Watermark on every page
th.watermark = Watermark(enabled=True, text="DRAFT", opacity=0.08, rotation=-45)

# In-flow banner at the top of page 1
th.banner = Banner(text="RESTRICTED — DO NOT DISTRIBUTE", color="FFFFFF", bg="B00020")

m = MarkItUp(theme=th)
out = m.convert("legal-contract.md", f"{out_dir}/legal-branded.docx")
print(f"✅ {out}")

# Quick verification
from docx import Document
doc = Document(out)
sec = doc.sections[0]
hdr = ' '.join(r.text for r in sec.header.paragraphs[0].runs)[:80]
ftr = ' '.join(r.text for r in sec.footer.paragraphs[0].runs)[:80]
has_wm = 'MarkItUpWatermark' in sec.header.paragraphs[0]._p.xml
print(f"   header: {hdr}")
print(f"   footer: {ftr}")
print(f"   watermark: {has_wm}")

## 4. Batch Convert All Samples

Run the `convert_all.py` script to process every sample in one shot.

In [ ]:
# Run the batch converter programmatically
import subprocess, sys

result = subprocess.run(
    [sys.executable, "convert_all.py", "--docx-only"],
    capture_output=True, text=True, cwd="."
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

# Show what was generated
for f in sorted(os.listdir(out_dir)):
    size_kb = os.path.getsize(f"{out_dir}/{f}") / 1024
    print(f"  {size_kb:6.0f} KiB  {f}")

## 5. Font Explorer

See what's installed on this machine and browse the cross-platform safe families.